In [ ]:
from pathlib import Path
from typing import List, Dict, Tuple, Union
from collections import defaultdict
import json


import numpy as np
import pandas as pd
import mne
import h5py
from tqdm.auto import tqdm

from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from helpers import compute_ceiling_splithalf, compute_ceiling_variancebased

In [ ]:
SUBJECTS = [f"P{i}" for i in range(1, 5)]

# ONSET = 0.1 # Experiments start at -0.1 seconds
# POINTS_PER_SEC = 200
# SHIFT = int(ONSET * POINTS_PER_SEC)
# TIME_POINTS = (int(0.0*POINTS_PER_SEC + SHIFT), int(1.3*POINTS_PER_SEC + SHIFT))  # Available: -0.1 to 1.3, selecting  0 ro 1.3

TIME_POINTS = [0, 1.3]


ROIS = {
    "occipital": "MLO",
    "parietal": "MLP",
    "temporal": "MLT",
    "frontal": "MLF",
    "central": "MLC",
}

ROIS_EXTENDED = [
    "occipital",
    "parietal",
    "temporal",
    "frontal",
    "central",
    "occipital_parietal",
    "whole_brain"
]

SUBJECTS, TIME_POINTS

In [ ]:
ds_dir = "${MBS_THINGS_RAW_DIR}/meg/LOCAL/ocontier/thingsmri/openneuro/THINGS-data/THINGS-MEG/ds004212/derivatives/preprocessed"
ds_dir = Path(ds_dir)

# !ls {ds_dir}

In [ ]:
df = pd.read_csv(ds_dir / "eyes_epoched_cleaned_P1_S10.csv", index_col=0)
df

In [ ]:
subject = "P2"

fif = mne.read_epochs(ds_dir / f"preprocessed_{subject}-epo.fif")

In [ ]:
fif.crop(tmin=0.0, tmax=1.3).get_data().shape  # (n_epochs, n_sensors, n_timepoints)

In [ ]:
# print(fif.get_data().shape)

In [ ]:
# print(fif._metadata.shape)

In [ ]:
# print(fif._metadata.image_path.nunique())

In [ ]:
# print(fif._metadata.image_path.nunique())

In [ ]:
# fif._metadata.trial_type.value_counts()

In [ ]:
# fif.events

In [ ]:
fif[0]

In [ ]:
subject_meg_data = {}
subject_metadata = {}

for subj in tqdm(SUBJECTS):
    print(f"Processing subject {subj}...")
    epochs = mne.read_epochs(ds_dir / f"preprocessed_{subj}-epo.fif")
    epochs = epochs.crop(tmin=TIME_POINTS[0], tmax=TIME_POINTS[1])
    epochs = epochs.resample(sfreq=100)  # downsample to 100 Hz
    channel_names = epochs.info["ch_names"]
    time_points = epochs.times  # in seconds
    
    
    # get all channel names that start with O, MLT, or MLP
    # occipital, temporal, parietal
    # sel_chs = [ch for ch in epochs.ch_names if ch.startswith(("MLO", "MLT", "MLP"))]
    # sel_chs = [ch for ch in channel_names if ch.startswith(("MLO"))]

    # keep only those channels (in-place)
    # epochs.pick(sel_chs)
    
    subject_meg_data[subj] = epochs.get_data()  # shape: (n_epochs, n_sensors, n_timepoints) # stimulus x channel x timepoints
    subject_metadata[subj] = epochs._metadata

In [ ]:
raise

In [ ]:
channels_by_roi = {}
channels_by_roi_masks = {}
for roi_name in tqdm(ROIS_EXTENDED):
    if roi_name == "whole_brain":
        channels_by_roi[roi_name] = channel_names
        channels_by_roi_masks[roi_name] = np.ones_like(channel_names, dtype=bool)
    elif roi_name == "occipital_parietal":
        channels_by_roi[roi_name] = [
            ch for ch in channel_names if (ch.startswith("MLO") or ch.startswith("MLP"))
        ]
        channels_by_roi_masks[roi_name] = np.where(
            np.array([(ch.startswith("MLO") or ch.startswith("MLP")) for ch in channel_names])
        )[0]
    else:
        roi_prefix = ROIS[roi_name]
        channels_by_roi[roi_name] = [
            ch for ch in channel_names if ch.startswith(roi_prefix)
        ]
        channels_by_roi_masks[roi_name] = np.where(
            np.array([ch.startswith(roi_prefix) for ch in channel_names])
        )[0]
    
    print(f"ROI {roi_name}: {len(channels_by_roi[roi_name])} channels.")

remaining_channels = set(channel_names)
for ch_list in channels_by_roi.values():
    remaining_channels -= set(ch_list)
print(f"Channels not assigned to any ROI: {remaining_channels}")


In [ ]:
subject_train_data_reps, subject_test_data_reps = {}, {}
subject_train_metadata, subject_test_metadata = {}, {}

for subj in tqdm(SUBJECTS):
    # Get MEG data and metadata for the subject
    meg_data = subject_meg_data[subj]
    metadata = subject_metadata[subj]
    
    # Split indices for train and test trials
    train_indices = metadata[metadata.trial_type == "exp"].index
    test_indices = metadata[metadata.trial_type == "test"].index
    
    # Split MEG data
    meg_data_train = meg_data[train_indices]
    meg_data_test = meg_data[test_indices]
    
    # Split metadata
    metadata_train = metadata.loc[train_indices]
    metadata_test = metadata.loc[test_indices]

    # Sort indices
    train_indices_sorted = metadata_train.image_nr.argsort()
    test_indices_sorted = metadata_test.image_nr.argsort()
    
    # Sort MEG data
    meg_data_train_sorted = meg_data_train[train_indices_sorted]
    meg_data_test_sorted = meg_data_test[test_indices_sorted]
    
    # Sort metadata
    metadata_train_sorted = metadata_train.iloc[train_indices_sorted]
    metadata_test_sorted = metadata_test.iloc[test_indices_sorted]
    
    subject_train_metadata[subj] = metadata_train_sorted
    subject_test_metadata[subj] = metadata_test_sorted
    
    subject_train_data_reps[subj] = {}
    subject_test_data_reps[subj] = {}
    for roi_name, roi_mask in tqdm(channels_by_roi_masks.items(), leave=False, desc=f"Subject {subj} ROIs"):
        # Reshape MEG data to (n_sensors, n_timepoints, n_images) and then apply ROI mask
        meg_data_train_roi = meg_data_train_sorted.transpose((1, 2, 0))[roi_mask]
        meg_data_test_roi = meg_data_test_sorted.transpose((1, 2, 0))[roi_mask]

        #  Reshape to (n_sensors, n_timepoints, n_images, n_repetitions)
        meg_data_train_reps = meg_data_train_roi.reshape((meg_data_train_roi.shape[0], meg_data_train_roi.shape[1], -1, 1))
        meg_data_test_reps = meg_data_test_roi.reshape((meg_data_test_roi.shape[0], meg_data_test_roi.shape[1], -1, 12))


        subject_train_data_reps[subj][roi_name] = meg_data_train_reps.transpose((2, 0, 1, 3))
        subject_test_data_reps[subj][roi_name] = meg_data_test_reps.transpose((2, 0, 1, 3))

In [ ]:
for subj in SUBJECTS:
    for roi_name in ROIS_EXTENDED:
        print(f"Subject {subj}, ROI {roi_name}:")
        print(f"  Train data shape: {subject_train_data_reps[subj][roi_name].shape}")  # (n_images, n_sensors, n_timepoints, n_repetitions)
        print(f"  Test data shape: {subject_test_data_reps[subj][roi_name].shape}")    # (n_images, n_sensors, n_timepoints, n_repetitions)
        print(f"  Train metadata shape: {subject_train_metadata[subj].shape}")
        print(f"  Test metadata shape: {subject_test_metadata[subj].shape}")
    break

In [ ]:
noise_ceilings_variancebased, noise_ceilings_splithalf = {}, {}

for subj in tqdm(SUBJECTS):
    
    noise_ceilings_variancebased[subj] = {}
    noise_ceilings_splithalf[subj] = {}
    for roi_name in tqdm(ROIS_EXTENDED, leave=False, desc=f"Subject {subj} ROIs"):
        data = subject_test_data_reps[subj][roi_name]  # shape: (n_images, n_sensors, n_timepoints, n_repetitions)

        data = data.transpose((1, 2, 0, 3))  # shape: (sensors, timepoints, images, repetitions)
        # Compute noise ceilings

        noise_ceilings_variancebased[subj][roi_name] = compute_ceiling_variancebased(data)
        noise_ceilings_splithalf[subj][roi_name] = compute_ceiling_splithalf(data).mean(axis=-1)

In [ ]:
for roi_name in ROIS_EXTENDED:
    fig, axes = plt.subplots(2, 2, figsize=(20, 8), dpi =100)

    for i, subj in enumerate(SUBJECTS):
        ax = axes[i // 2, i % 2]

        time = np.linspace(TIME_POINTS[0], TIME_POINTS[1], noise_ceilings_variancebased[subj][roi_name].shape[1])
        sns.lineplot(x=time, y=noise_ceilings_variancebased[subj][roi_name].mean(axis=0), ax=ax, label='Variance-Based')
        sns.lineplot(x=time, y=noise_ceilings_splithalf[subj][roi_name].mean(axis=0), ax=ax, label='Split-Half')


        ax.set_title(f"Subject {subj}")
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('Noise Ceiling (Pearson r)')

        # put a horizontal line at maximum of of all ceilings
        max_ceiling_test = max(noise_ceilings_variancebased[subj][roi_name].mean(axis=0).max(), noise_ceilings_splithalf[subj][roi_name].mean(axis=0).max())
        ax.axhline(max_ceiling_test, color='k', linestyle='--', label='Max Ceiling (Test) @ {:.2f}'.format(max_ceiling_test))
        ax.axhline(0, color='k', linestyle='--')

        ax.legend(loc='upper right')

    plt.suptitle(f"Noise Ceilings for ROI: {roi_name}", fontsize=20, fontweight='bold')
    plt.tight_layout()

In [ ]:
# subject_train_data_avg = {
#     subj: data.mean(axis=-1)  # Average over repetitions
#     for subj, data in subject_train_data_reps.items()
# }

# subject_test_data_avg = {
#     subj: data.mean(axis=-1)  # Average over repetitions
#     for subj, data in subject_test_data_reps.items()
# }


subject_train_data_avg = {}
subject_test_data_avg = {}
for subj in tqdm(SUBJECTS):
    subject_train_data_avg[subj] = {}
    subject_test_data_avg[subj] = {}
    for roi_name in tqdm(ROIS_EXTENDED, leave=False, desc=f"Subject {subj} ROIs"):
        subject_train_data_avg[subj][roi_name] = subject_train_data_reps[subj][roi_name].mean(axis=-1)  # Average over repetitions
        subject_test_data_avg[subj][roi_name] = subject_test_data_reps[subj][roi_name].mean(axis=-1)  # Average over repetitions
    

In [ ]:
for subj in SUBJECTS:
    for roi_name in ROIS_EXTENDED:
        print(f"Subject {subj} - ROI {roi_name}:")
        print(f"  Train data shape: {subject_train_data_avg[subj][roi_name].shape}")  # (n_images, n_sensors, n_timepoints)
        print(f"  Test data shape: {subject_test_data_avg[subj][roi_name].shape}")    # (n_images, n_sensors, n_timepoints)

In [ ]:
metadata_train, metadata_test = subject_train_metadata[SUBJECTS[0]], subject_test_metadata[SUBJECTS[0]]

stimulus_images_train, stimulus_images_test = metadata_train.image_path.to_numpy(), metadata_test.image_path.to_numpy()
stimulus_images_test = stimulus_images_test.reshape(-1, 12)[:, 0]

for subj in SUBJECTS[1:]:
    assert np.array_equal(stimulus_images_train, subject_train_metadata[subj].image_path.to_numpy())
    assert np.array_equal(stimulus_images_test, subject_test_metadata[subj].image_path.to_numpy().reshape(-1, 12)[:, 0])

train_stimulus_ids = [f.split("/", 1)[1] for f in stimulus_images_train.tolist()]
test_stimulus_ids = [f.split("/", 1)[1] for f in stimulus_images_test.tolist()]
test_stimulus_ids = [f"{f.rsplit('_', 1)[0]}/{f}" for f in test_stimulus_ids]

len(train_stimulus_ids), len(test_stimulus_ids), train_stimulus_ids[0], test_stimulus_ids[4]

### Concatenate data

In [ ]:
processed_data = {
    "train" :
        {
            "stimulus_ids": train_stimulus_ids,
            "neural_data": subject_train_data_avg,
        },
    "test" :
        {
            "stimulus_ids": test_stimulus_ids,
            "neural_data": subject_test_data_avg,
        },
    "noise_ceilings": noise_ceilings_variancebased,
}

### Metadata

In [ ]:
metadata = {
    "desc": """
    The neural data is from the THINGS MEG, recorded from 4 humans.
    All neural data is averaged across trials for each image.
    """
}
metadata_str = json.dumps(metadata, indent=2)
metadata_str = json.dumps(metadata, indent=2).encode('utf-8')

### Save to disk

In [ ]:
data_dir = '${MBS_DATA_PREP_OUTPUT_DIR}'
filename = f'things_meg.h5'

data_dir = Path(data_dir)
data_path = data_dir / filename

if not data_dir.exists():
    data_dir.mkdir(parents=False, exist_ok=False)

In [ ]:
with h5py.File(data_path, 'w') as f:
    for split in ['train', 'test']:
        f.create_dataset(f"{split}/stimulus_ids", data=processed_data[split]['stimulus_ids'])

        for subj in tqdm(SUBJECTS):
            for roi_name in ROIS_EXTENDED:
                f.create_dataset(f"{split}/neural_data/{subj}/{roi_name}", data=processed_data[split]['neural_data'][subj][roi_name])

    for subj in SUBJECTS:
        for roi_name in ROIS_EXTENDED:
            f.create_dataset(f"noise_ceilings/{subj}/{roi_name}", data=processed_data['noise_ceilings'][subj][roi_name])

    f.attrs['metadata'] = metadata_str
    f.attrs['rois'] = ROIS_EXTENDED
    f.attrs['subjects'] = list(SUBJECTS)
    f.attrs['splits'] = ['train', 'test']
    f.attrs['max_nc'] = 100
    f.attrs['time_points'] = time_points.tolist()
    f.attrs['channels'] = channel_names
    f.close()


In [ ]:
loaded_data = defaultdict(dict)
with h5py.File(data_path, 'r') as f:
    splits = f.attrs['splits']
    subjects = f.attrs['subjects']
    rois = f.attrs['rois']
    for split in splits:
        loaded_data[split]['stimulus_ids'] = f[split]['stimulus_ids'][()]
        
        loaded_data[split]['neural_data'] = {}
        for subj in subjects:
            loaded_data[split]['neural_data'][subj] = {}
            for roi in rois:
                loaded_data[split]['neural_data'][subj][roi] = f[split]['neural_data'][subj][roi][()]
                
    for subj in subjects:
        loaded_data['noise_ceilings'][subj] = {}
        for roi in rois:
            loaded_data['noise_ceilings'][subj][roi] = f['noise_ceilings'][subj][roi][()]
            


In [ ]:
loaded_data['test'].keys()
loaded_data['test']['neural_data']['P1']['whole_brain'].shape, loaded_data['noise_ceilings']['P1']['whole_brain'].shape